In [0]:
import os
import time
import requests
import zipfile
import urllib3
import hashlib
from pathlib import Path

In [0]:
# Exceção temporária para o ambiente de desenvolvimento
urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

# Configurações
anos = range(2020, 2025)

base_url = (
    "https://download.inep.gov.br/microdados/"
    "microdados_censo_da_educacao_superior_{}.zip"
)

volume = "/Volumes/workspace/bronze/raw_inep_superior"

max_tentativas = 5

# Criar volume, caso ainda não exista
spark.sql("""
    CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_inep_superior
""")

# Processar os anos sequencialmente
for ano in anos:

    print(f"\n{'=' * 55}")
    print(f"Processando Censo da Educação Superior: {ano}")
    print(f"{'=' * 55}")

    url = base_url.format(ano)

    pasta_destino = Path(f"{volume}/{ano}")
    pasta_destino.mkdir(parents=True, exist_ok=True)

    arquivo_destino = (
        pasta_destino / f"microdados_inep_{ano}.zip"
    )

    arquivo_temporario = (
        pasta_destino / f"microdados_inep_{ano}.zip.part"
    )

    # Evitar download se o ZIP existente for válido
    if arquivo_destino.exists():
        with zipfile.ZipFile(arquivo_destino, "r") as zip_ref:
            arquivo_valido = zip_ref.testzip() is None

        if arquivo_valido:
            print(f"Arquivo já existente e íntegro: {arquivo_destino}")
            continue

        print("Arquivo existente inválido. Será baixado novamente.")

    # Download com tentativas e validação
    for tentativa in range(1, max_tentativas + 1):

        try:
            print(f"Tentativa {tentativa}/{max_tentativas}")

            with requests.get(
                url,
                stream=True,
                timeout=(30, 120),
                allow_redirects=True,
                verify=False  # Exceção temporária: TLS não verificado
            ) as resposta:

                resposta.raise_for_status()

                tamanho_esperado = resposta.headers.get(
                    "Content-Length"
                )

                with open(arquivo_temporario, "wb") as arquivo:
                    for bloco in resposta.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if bloco:
                            arquivo.write(bloco)

            tamanho_baixado = arquivo_temporario.stat().st_size

            # Conferir tamanho informado pelo servidor
            if tamanho_esperado is not None:
                if tamanho_baixado != int(tamanho_esperado):
                    raise ValueError(
                        f"Tamanho divergente: {tamanho_baixado} "
                        f"!= {tamanho_esperado}"
                    )

            if tamanho_baixado == 0:
                raise ValueError("Arquivo vazio.")

            # Validar estrutura e integridade do ZIP
            with zipfile.ZipFile(arquivo_temporario, "r") as zip_ref:
                arquivo_corrompido = zip_ref.testzip()

                if arquivo_corrompido is not None:
                    raise ValueError(
                        f"Arquivo corrompido: {arquivo_corrompido}"
                    )

            # Publicar arquivo definitivo
            os.replace(arquivo_temporario, arquivo_destino)

            print(f"Download concluído: {ano}")
            print(f"Destino: {arquivo_destino}")
            print(
                f"Tamanho: "
                f"{tamanho_baixado / 1024**2:.2f} MiB"
            )

            break

        except (
            requests.RequestException,
            ValueError,
            zipfile.BadZipFile,
            OSError
        ) as erro:

            print(f"Erro: {type(erro).__name__}: {erro}")

            # Excluir download incompleto
            if arquivo_temporario.exists():
                arquivo_temporario.unlink()

            if tentativa == max_tentativas:
                raise RuntimeError(
                    f"Falha no download do ano {ano}"
                ) from erro

            time.sleep(5 * tentativa)

print("\nProcessamento de todos os anos concluído!")

In [0]:
volume = Path("/Volumes/workspace/bronze/raw_inep_superior")

inventario = []

for ano in anos:

    arquivo = volume / str(ano) / f"microdados_inep_{ano}.zip"

    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")

    sha256 = hashlib.sha256()

    with open(arquivo, "rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(bloco)

    inventario.append({
        "ano": ano,
        "arquivo": str(arquivo),
        "tamanho_bytes": arquivo.stat().st_size,
        "sha256": sha256.hexdigest()
    })

display(spark.createDataFrame(inventario))

print(f"Arquivos identificados: {len(inventario)}")